In [27]:
import pandas as pd
import numpy as np
import importlib
import config
import sys
sys.path.insert(0, '../scripts')
from scripts.remove_non_trading_days import remove_non_trading_days
importlib.reload(config)
import tabulate

from config import DAILY_SENTIMENT,PRICE_FEATURES,MODEL_DATASET

In [2]:
sdf = pd.read_parquet(DAILY_SENTIMENT)
sdf = sdf.rename(columns={'trading_day': 'date'})

In [3]:
pdf = pd.read_parquet(PRICE_FEATURES)

In [4]:
#Ensure again that there are no non-trading days in either frame
sdf = remove_non_trading_days(sdf,'date')
pdf = remove_non_trading_days(pdf,'date')

There are currently 1921 days in your df
There are 1921 days in the NYSE schedule
Therefore we remove 0 days from the df
df shape after cleaning:(2433754, 11)
There are currently 1921 days in your df
There are 1921 days in the NYSE schedule
Therefore we remove 0 days from the df
df shape after cleaning:(2408260, 17)


In [5]:
print(pdf.duplicated(subset=['ISIN', 'date']).sum())   # expect 0
print(sdf.duplicated(subset=['ISIN', 'date']).sum())   # expect 0

0
0


In [6]:
print(pdf['date'].dtype, sdf['date'].dtype)   # both datetime64[ns]

datetime64[ns] datetime64[ns]


In [7]:
df = pdf.merge(sdf, on=['ISIN', 'date'], how='left', validate='one_to_one')
assert len(df) == len(pdf)

In [8]:
coverage = df['sentiment_score'].notna().mean()
print(f'Rows with sentiment: {coverage:.1%}')

Rows with sentiment: 100.0%


In [9]:
print(len(sdf), len(pdf))                          # expect equal
print((sdf['sentiment_volume'] > 0).mean())        # proportion of genuine news days

2433754 2408260
0.32560151929899245


In [10]:
df['has_news'] = df['sentiment_volume'] > 0

In [11]:
df.groupby('date')['universe_sentiment_score'].nunique().max()   # expect 1

np.int64(1)

In [12]:
orphans = sdf.merge(pdf[['ISIN', 'date']], on=['ISIN', 'date'],
                    how='left', indicator=True)
orphans = orphans[orphans['_merge'] == 'left_only']
print(orphans['ISIN'].nunique())
print(orphans['date'].min(), orphans['date'].max())
print(orphans.groupby('ISIN').size().sort_values(ascending=False).head())

301
2017-12-15 00:00:00 2025-07-28 00:00:00
ISIN
US3693001089    1792
US12673P1057    1502
US8679141031    1402
US87403A1079    1267
US2782651036    1051
dtype: int64


In [13]:
universe_isins = set(pdf['ISIN'].unique())
orphans['isin_in_universe'] = orphans['ISIN'].isin(universe_isins)

date_min, date_max = pdf['date'].min(), pdf['date'].max()
orphans['date_out_of_window'] = (orphans['date'] < date_min) | (orphans['date'] > date_max)

print(orphans.groupby(['isin_in_universe', 'date_out_of_window']).size())

isin_in_universe  date_out_of_window
True              False                 25494
dtype: int64


In [14]:
bounds = pdf.groupby('ISIN')['date'].agg(first_px='min', last_px='max')
o = orphans.merge(bounds, on='ISIN', how='left')

before = o['date'] < o['first_px']
after  = o['date'] > o['last_px']
inside = ~before & ~after

print(f'Before first price date: {before.sum()}')
print(f'After last price date:  {after.sum()}')
print(f'Inside price history:   {inside.sum()}')

Before first price date: 1008
After last price date:  24477
Inside price history:   9


In [15]:
print(o[inside][['ISIN', 'date', 'sentiment_volume']])

               ISIN       date  sentiment_volume
16261  US82669G1040 2023-03-16          2.708050
16262  US82669G1040 2023-03-17          2.197225
16263  US82669G1040 2023-03-20          3.218876
16264  US82669G1040 2023-03-21          2.302585
16265  US82669G1040 2023-03-22          0.000000
16266  US82669G1040 2023-03-23          1.609438
16267  US82669G1040 2023-03-24          1.386294
16268  US82669G1040 2023-03-27          1.945910
19702  US29605J1060 2022-04-04          0.000000


In [16]:
isin, d = 'US...', pd.Timestamp('2023-03-16')   # from the output above
mask = (pdf['ISIN'] == isin) & (pdf['date'].between(d - pd.Timedelta(days=7),
                                                    d + pd.Timedelta(days=7)))
print(pdf[mask][['date']].to_string())

Empty DataFrame
Columns: [date]
Index: []


In [17]:
rows_before = len(df)
df = df[df['date'].between('2018-01-04', '2025-06-30')]
print(f'Trimmed {rows_before - len(df):,} rows; {len(df):,} remain')
print(df['date'].min(), df['date'].max())

Trimmed 38,071 rows; 2,370,189 remain
2018-01-04 00:00:00 2025-06-30 00:00:00


In [18]:
# Incumbents should have a populated 20d MA on the first sample day
first_day = df[df['date'] == df['date'].min()]
print(f"MA coverage on first day: {first_day['r0_z20'].notna().mean():.1%}")

MA coverage on first day: 5.7%


In [26]:
df[df['ISIN']=='US5949181045'].head(20)

,ISIN,bb_tcm,date,Sector,r0,r_1,r_5_2,r_10_6,pvma,r0_z20,...,fwd_r5_sec,sentiment_score,sentiment_volume,sentiment_score_z,sentiment_volume_z,story_count,universe_story_count,universe_sentiment_score,universe_sentiment_volume,has_news
1455404,US5949181045,MSFT US,2018-01-04,Information Technology,0.008763,0.004643,0.002796,-0.001399,0.020031,0.633987,...,-0.007191,0.000000,2.079442,0.421925,1.156275,7.0,1729.0,-0.106449,7.234898,True
1455405,US5949181045,MSFT US,2018-01-05,Information Technology,0.012322,0.008763,0.007323,0.002336,0.029189,1.014531,...,-0.001777,0.037602,3.258097,0.607059,2.743254,27.0,1500.0,-0.128908,7.077498,True
1455406,US5949181045,MSFT US,2018-01-08,Information Technology,0.001020,0.012322,0.018188,0.000468,0.026835,-0.244111,...,-0.001330,-0.048810,3.295837,-0.018263,2.444915,28.0,2233.0,0.014259,7.471932,True
1455407,US5949181045,MSFT US,2018-01-09,Information Technology,-0.000680,0.001020,0.025728,0.005132,0.023796,-0.469911,...,-0.000464,-0.663518,1.945910,-1.106281,1.237223,8.0,1340.0,-0.017473,6.946014,True
1455408,US5949181045,MSFT US,2018-01-10,Information Technology,-0.004544,-0.000680,0.022105,0.011063,0.017749,-0.889098,...,-0.000656,1.000000,0.693147,1.949820,-0.793859,1.0,1168.0,0.006043,6.795706,True
1455409,US5949181045,MSFT US,2018-01-11,Information Technology,0.002956,-0.004544,0.012662,0.016202,0.019258,0.194333,...,0.001920,0.228334,1.386294,0.947292,0.410418,3.0,1245.0,-0.064073,6.801283,True
1455410,US5949181045,MSFT US,2018-01-12,Information Technology,0.017110,0.002956,-0.004204,0.028407,0.033911,2.091170,...,0.011774,-0.058677,2.944439,0.200929,2.499458,20.0,1259.0,-0.044651,6.858565,True
1455411,US5949181045,MSFT US,2018-01-16,Information Technology,-0.014049,0.017110,-0.002268,0.031529,0.017751,-2.011022,...,0.019276,0.233059,2.639057,0.390899,2.014761,41.0,2560.0,0.057101,7.448916,True
1455412,US5949181045,MSFT US,2018-01-17,Information Technology,0.020058,-0.014049,0.015522,0.026068,0.035916,2.059943,...,0.021752,0.151669,1.945910,0.356614,1.391685,8.0,1582.0,-0.053514,7.033506,True
1455413,US5949181045,MSFT US,2018-01-18,Information Technology,-0.000444,0.020058,0.006017,0.016880,0.033335,-0.286238,...,0.028995,0.139241,2.197225,0.257010,1.705922,8.0,1642.0,-0.018943,7.045777,True


In [23]:
df.rename(columns={'Sector_x':'Sector'}, inplace=True)

In [25]:
df.drop(columns=['Sector_y'], inplace=True)

In [28]:
df.to_parquet(MODEL_DATASET, index=False)